# 05 Monitoring Story And Dashboard

## 목표
- 튜닝이 끝난 규칙을 운영 관점에서 해석한다.
- `baseline / balanced / conservative` 세 규칙을 대시보드 관점에서 비교한다.
- 이 프로젝트를 반도체 공정 모니터링 문제로 어떻게 번역할지 스토리라인을 정리한다.

## 이 노트북의 산출물
- `monitoring_kpi_summary.csv`
- `daily_monitoring_summary.csv`
- `top_balanced_alert_days.csv`
- `story_translation_table.csv`
- `portfolio_storyline.csv`

## 작성 원칙
- 코드 셀 안에는 어려운 용어와 정의 이유를 한글 주석으로 남긴다.
- 각 코드 셀 아래에는 결과 해석과 다음 액션을 2~3줄로 정리한다.


In [ ]:
import os
from pathlib import Path
import warnings

# 결과물 폴더를 먼저 고정해두면 대시보드용 표와 스토리 표를 한곳에 정리할 수 있다.
PROJECT_DIR = Path("/Users/chankyulee/Desktop/ModuLABS/05_TimeSeries/Projects/FTS_Projects")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "05_monitoring_story_and_dashboard"
PREV_OUTPUT_DIR = PROJECT_DIR / "outputs" / "04_threshold_tuning_and_rules"
MPLCONFIG_DIR = PROJECT_DIR / ".mplconfig"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
sns.set_theme(style="whitegrid", context="notebook")

print(f"PREV_OUTPUT_DIR exists: {PREV_OUTPUT_DIR.exists()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


### 결과 해석
- 이번 노트북은 앞선 모델링 결과를 읽어와 스토리와 대시보드로 정리하는 단계이므로, 이전 출력 폴더와 현재 출력 폴더를 분리해두는 것이 중요합니다. 이렇게 해두면 분석 산출물과 발표용 정리 자료가 뒤섞이지 않습니다.
- 문제는 노트북이 길어질수록 파일이 흩어지기 쉽다는 점인데, 원인은 단계별 결과 저장 위치가 다르기 때문입니다. 출력 경로를 명시적으로 나눠두면 발표 준비 단계에서 자료를 다시 찾느라 시간을 쓰지 않아도 됩니다.


## 1. 최종 규칙과 KPI 불러오기

이 단계에서는 앞선 노트북에서 선택한 최종 규칙과 test 성능을 다시 불러와 운영 관점의 KPI로 재정리합니다.


In [ ]:
# 이전 노트북 결과를 그대로 읽어오면 중복 계산 없이 스토리 정리에 집중할 수 있다.
selected_configs = pd.read_csv(PREV_OUTPUT_DIR / "selected_rule_configs.csv")
test_rule_comparison = pd.read_csv(PREV_OUTPUT_DIR / "test_rule_comparison.csv")
test_rule_predictions = pd.read_csv(PREV_OUTPUT_DIR / "test_rule_predictions.csv", parse_dates=["time"])
rule_window_sample = pd.read_csv(PREV_OUTPUT_DIR / "rule_window_sample.csv", parse_dates=["time"])

# baseline을 기준점으로 두고 각 tuned rule이 얼마나 개선됐는지 차이를 계산한다.
baseline_row = test_rule_comparison.loc[
    test_rule_comparison["config_name"] == "baseline_zscore"
].iloc[0]

monitoring_kpi_summary = test_rule_comparison.copy()
monitoring_kpi_summary["point_f1_delta_vs_baseline"] = (
    monitoring_kpi_summary["point_f1"] - baseline_row["point_f1"]
)
monitoring_kpi_summary["point_fpr_reduction_vs_baseline"] = (
    1 - monitoring_kpi_summary["point_fpr"] / baseline_row["point_fpr"]
)
monitoring_kpi_summary["false_alerts_per_day_reduction_vs_baseline"] = (
    1 - monitoring_kpi_summary["false_alerts_per_day"] / baseline_row["false_alerts_per_day"]
)
monitoring_kpi_summary["event_f1_delta_vs_baseline"] = (
    monitoring_kpi_summary["event_f1"] - baseline_row["event_f1"]
)

# 운영 기본안은 balanced, 운영 보수안은 conservative로 명시한다.
monitoring_kpi_summary["operating_role"] = monitoring_kpi_summary["config_name"].map(
    {
        "baseline_zscore": "비교 기준",
        "balanced_selected": "기본 운영안",
        "conservative_selected": "보수 운영안",
    }
)

monitoring_kpi_summary.to_csv(OUTPUT_DIR / "monitoring_kpi_summary.csv", index=False)

display(selected_configs)
display(monitoring_kpi_summary)


### 결과 해석
- test 기준으로 `balanced_selected`는 baseline 대비 `point_f1`가 약 **+0.0187** 좋아졌고, `point_fpr`는 약 **43.95%**, `false_alerts/day`는 약 **47.27%** 줄였습니다. 반면 `conservative_selected`는 `false_alerts/day`를 약 **72.88%** 줄이지만 `event_f1`는 더 내려갑니다.
- 따라서 기본 추천안은 `balanced`, 운영 보수안은 `conservative`가 자연스럽습니다. 인사이트는 '좋은 모델 1개'를 고르는 것보다, 운영 목적에 따라 `탐지형 룰`과 `보수형 룰`을 나눠 제안하는 편이 훨씬 실무적이라는 점입니다.


## 2. Daily Monitoring Summary

경보 시스템은 결국 하루 단위 운영 지표가 중요하므로, 일별 이벤트 수와 경보 수를 정리합니다.


In [ ]:
# 운영 관점에서는 minute 단위보다 하루 단위 집계가 훨씬 읽기 쉽다.
test_rule_predictions["date"] = test_rule_predictions["time"].dt.date

daily_monitoring_summary = (
    test_rule_predictions
    .groupby("date")[["pseudo_event", "baseline_zscore", "balanced_selected", "conservative_selected"]]
    .sum()
    .reset_index()
)

# '그날 이벤트가 있었는데 경보가 0개였는가'는 운영 누락 관점에서 의미 있는 지표다.
daily_monitoring_summary["balanced_missed_day"] = (
    (daily_monitoring_summary["pseudo_event"] > 0)
    & (daily_monitoring_summary["balanced_selected"] == 0)
)
daily_monitoring_summary["conservative_missed_day"] = (
    (daily_monitoring_summary["pseudo_event"] > 0)
    & (daily_monitoring_summary["conservative_selected"] == 0)
)

top_balanced_alert_days = (
    daily_monitoring_summary
    .sort_values("balanced_selected", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

daily_monitoring_summary.to_csv(OUTPUT_DIR / "daily_monitoring_summary.csv", index=False)
top_balanced_alert_days.to_csv(OUTPUT_DIR / "top_balanced_alert_days.csv", index=False)

daily_kpi = pd.DataFrame(
    {
        "metric": [
            "num_days",
            "avg_daily_pseudo_event",
            "avg_daily_baseline_alert",
            "avg_daily_balanced_alert",
            "avg_daily_conservative_alert",
            "max_daily_balanced_alert",
            "balanced_missed_days",
            "conservative_missed_days",
        ],
        "value": [
            len(daily_monitoring_summary),
            daily_monitoring_summary["pseudo_event"].mean(),
            daily_monitoring_summary["baseline_zscore"].mean(),
            daily_monitoring_summary["balanced_selected"].mean(),
            daily_monitoring_summary["conservative_selected"].mean(),
            daily_monitoring_summary["balanced_selected"].max(),
            int(daily_monitoring_summary["balanced_missed_day"].sum()),
            int(daily_monitoring_summary["conservative_missed_day"].sum()),
        ],
    }
)

display(daily_kpi)
display(top_balanced_alert_days)


### 결과 해석
- test 구간은 총 **204일**이고, 일평균 `pseudo_event`는 약 **8.82건**입니다. 일평균 경보 수는 `baseline 14.18건`, `balanced 8.64건`, `conservative 4.68건`으로 줄어들어 balanced가 실제 이벤트 규모에 가장 가깝게 내려왔습니다.
- `balanced`는 이벤트가 있었던 날 중 경보가 0건인 날이 **0일**, `conservative`는 **2일**이었습니다. 즉 balanced는 운영 누락을 거의 만들지 않으면서 경보량을 줄였고, conservative는 더 조용하지만 일부 약한 이벤트를 놓칠 수 있다는 점이 분명합니다.


## 3. Dashboard Mockup

앞선 결과를 실제 모니터링 화면처럼 읽을 수 있게 KPI, 일별 경보량, 상위 경보일, 이벤트 창 반응을 한 장에 묶어봅니다.


In [ ]:
# dashboard는 한 화면에서 '정량 KPI + 시계열 흐름 + 예시 이벤트'를 같이 보여줘야 한다.
fig = plt.figure(figsize=(18, 13))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.1])

# 1) KPI comparison bar chart
ax1 = fig.add_subplot(gs[0, 0])
kpi_plot = monitoring_kpi_summary[["config_name", "point_f1", "event_f1", "false_alerts_per_day"]].copy()
kpi_plot = kpi_plot.set_index("config_name")
kpi_plot[["point_f1", "event_f1"]].plot(kind="bar", ax=ax1)
ax1.set_title("Rule KPI Comparison")
ax1.tick_params(axis="x", rotation=20)
ax1.set_ylabel("score")

# false alert/day는 단위가 달라 같은 축에 넣지 않고 보조 축으로 따로 표시한다.
ax1b = ax1.twinx()
ax1b.plot(range(len(kpi_plot)), kpi_plot["false_alerts_per_day"], color="black", marker="o", linewidth=2)
ax1b.set_ylabel("false alerts / day")

# 2) daily monitoring line chart
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["pseudo_event"], label="pseudo_event", color="red", linewidth=1.2)
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["balanced_selected"], label="balanced_alert", color="royalblue", linewidth=1.2)
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["conservative_selected"], label="conservative_alert", color="green", linewidth=1.2)
ax2.set_title("Daily Monitoring Summary")
ax2.tick_params(axis="x", rotation=30)
ax2.legend()

# 3) top alert days bar chart
ax3 = fig.add_subplot(gs[1, 0])
top_plot = top_balanced_alert_days.copy()
top_plot["date"] = top_plot["date"].astype(str)
sns.barplot(data=top_plot, x="date", y="balanced_selected", color="royalblue", ax=ax3)
ax3.set_title("Top 10 Balanced Alert Days")
ax3.tick_params(axis="x", rotation=35)
ax3.set_ylabel("alert count")

# 4) example event window
ax4 = fig.add_subplot(gs[1, 1])
window_plot = rule_window_sample.fillna(0).copy()
ax4.plot(window_plot["time"], window_plot["raw_close"], color="black", linewidth=1.0, label="close")
ax4.scatter(window_plot.loc[window_plot["pseudo_event"] == 1, "time"], window_plot.loc[window_plot["pseudo_event"] == 1, "raw_close"], color="red", s=60, label="pseudo_event")
ax4.scatter(window_plot.loc[window_plot["balanced_selected"] == 1, "time"], window_plot.loc[window_plot["balanced_selected"] == 1, "raw_close"], color="royalblue", s=35, label="balanced")
ax4.scatter(window_plot.loc[window_plot["conservative_selected"] == 1, "time"], window_plot.loc[window_plot["conservative_selected"] == 1, "raw_close"], color="green", s=35, label="conservative")
ax4.set_title("Example Event Window")
ax4.tick_params(axis="x", rotation=30)
ax4.legend()

plt.tight_layout()
plt.show()


### 결과 해석
- 이 dashboard mockup에서 가장 중요한 메시지는 `balanced rule`이 baseline보다 경보량을 줄이면서도 KPI를 크게 해치지 않는다는 점입니다. 실제로 baseline의 `false_alerts/day≈8.56`이 balanced에서는 **4.51**로 내려가고, `point_f1`는 오히려 좋아졌습니다.
- 반면 conservative는 가장 조용하지만 이벤트 창에서 반응을 생략하는 경우가 늘어납니다. 즉 발표 시에는 `balanced를 기본 운영안`, `conservative를 야간/보수 모드`처럼 제안하면 훨씬 설득력 있는 운영 스토리가 됩니다.


## 4. 반도체 공정 모니터링 언어로 번역하기

이 프로젝트는 금융 데이터를 썼지만, 취업 포트폴리오에서는 제조/공정 모니터링 언어로 자연스럽게 번역하는 것이 중요합니다.


In [ ]:
# 같은 기술이라도 어떤 언어로 설명하느냐에 따라 포트폴리오의 직무 적합도가 달라진다.
story_translation_table = pd.DataFrame(
    {
        "crypto_project_term": [
            "급등/급락 이벤트",
            "거래량 burst",
            "변동성 spike",
            "false alert",
            "cooldown rule",
            "balanced rule",
            "conservative rule",
            "daily monitoring dashboard",
        ],
        "semiconductor_translation": [
            "공정 excursion 또는 이상 상태",
            "설비/센서 신호 급증",
            "공정 변동성 확대",
            "오경보",
            "중복 알람 억제 규칙",
            "탐지형 운영안",
            "보수형 운영안",
            "공정 상태 모니터링 대시보드",
        ],
        "interview_message": [
            "드문 이상 상태를 빠르게 포착하는 문제로 해석할 수 있습니다.",
            "평소 대비 급격한 상태 변화 감지로 설명할 수 있습니다.",
            "공정 안정성 저하 조기경보 문제로 연결할 수 있습니다.",
            "운영 가능성을 높이기 위해 오경보를 줄이는 것이 중요하다고 설명할 수 있습니다.",
            "같은 이상 이벤트에서 알람이 과도하게 반복되지 않도록 후처리했다고 말할 수 있습니다.",
            "이상 징후를 놓치지 않는 기본 운영안으로 제안할 수 있습니다.",
            "오경보 비용이 큰 라인에서 쓰는 보수 모드라고 설명할 수 있습니다.",
            "모델 성능을 운영 KPI와 함께 보여주는 화면이라고 설명할 수 있습니다.",
        ],
    }
)

portfolio_storyline = pd.DataFrame(
    {
        "section": [
            "문제 정의",
            "데이터 처리",
            "이상 이벤트 정의",
            "모델링",
            "운영 규칙 튜닝",
            "최종 제안",
        ],
        "portfolio_message": [
            "고빈도 시계열에서 이상 상태를 조기 탐지하는 모니터링 시스템을 설계했습니다.",
            "시간축 불연속성과 저유동성 구간을 분리해 실제 운영 가능한 데이터셋으로 정제했습니다.",
            "단순 point anomaly가 아니라 이벤트 창 개념을 도입해 실무형 평가 기준을 만들었습니다.",
            "rolling z-score, EWMA, Isolation Forest를 비교해 탐지형/보수형 trade-off를 확인했습니다.",
            "threshold와 cooldown을 validation에서 튜닝해 false alert/day를 크게 낮췄습니다.",
            "기본 운영안(balanced)과 보수 운영안(conservative)을 분리 제안해 실제 현업 적용 시나리오까지 정리했습니다.",
        ],
    }
)

story_translation_table.to_csv(OUTPUT_DIR / "story_translation_table.csv", index=False)
portfolio_storyline.to_csv(OUTPUT_DIR / "portfolio_storyline.csv", index=False)

display(story_translation_table)
display(portfolio_storyline)


### 결과 해석
- 이 표는 같은 프로젝트를 금융이 아니라 제조 모니터링 문제로 설명하기 위한 번역 가이드입니다. 특히 `false alert`, `cooldown`, `balanced/conservative rule` 같은 표현은 반도체 공정 모니터링 문맥으로 바꿔 말하기 쉽습니다.
- 취업 포트폴리오에서는 기술보다도 '어떤 문제를 풀었는가'를 직무 언어로 설명하는 것이 중요합니다. 따라서 면접에서는 `이상 상태 조기경보`, `오경보 최소화`, `운영 모드 분리`를 핵심 키워드로 가져가면 훨씬 설득력이 높아집니다.


## 5. 여기까지 하면 끝인가?

핵심 분석 흐름은 여기까지면 사실상 완성입니다. 다만 취업용 마무리로는 아래 두 가지만 추가하면 훨씬 좋습니다.

1. 5~7장 분량의 짧은 발표자료 또는 README
2. 면접 답변용 한 페이지 요약

즉, `01~05 노트북`까지가 프로젝트 본체이고, 이후는 전달력 강화를 위한 패키징 단계라고 보면 됩니다.
